# Stage 02 — Data Preparation

Load raw data, merge datasets, construct variables, write `data/dataset.parquet`.
Follow `skills/stage_02.md` for detailed instructions.

In [ ]:
from paths import *
import json
import numpy as np
import pandas as pd
import pyreadstat

spec = json.loads(PAPER_SPEC.read_text())
print('Identification type:', spec['identification']['type'])
print('Primary file:', spec['identification']['primary_data_file'])
print('Secondary datasets:', spec['identification'].get('secondary_datasets', []))

In [ ]:
# --- AGENT: load primary dataset ---
primary_file = RAW_DATA_DIR / spec['identification']['primary_data_file']
df, meta = pyreadstat.read_dta(str(primary_file))
print(f'Primary dataset shape: {df.shape}')
df.head()

In [ ]:
# --- AGENT: merge secondary datasets if needed ---
# For each secondary dataset in spec['identification']['secondary_datasets'],
# load and merge on the appropriate key. Document merges below.

# Example:
# df_ethnic, _ = pyreadstat.read_dta(str(RAW_DATA_DIR / 'ethnic.dta'))
# df = df.merge(df_ethnic, on='country_code', how='left')
# print(f'After merge: {df.shape}')
pass

In [ ]:
# --- AGENT: construct variables ---
# Follow spec['identification']['functional_form'] and 'controls'
# Log-transform outcome if needed
# Create squared terms if functional_form == 'quadratic'
# Create continent dummies if in fixed_effects
#
# Document each constructed variable with a comment
pass

In [ ]:
# --- Validate ---
key_vars = ([spec['identification']['outcome_variable'],
             spec['identification']['treatment_variable']] +
            ([spec['identification']['instrument']]
             if 'instrument' in spec['identification'] else []))

print('Missingness in key variables:')
print(df[key_vars].isnull().sum())
print(f'\nTotal rows with all key vars present: {df[key_vars].dropna().shape[0]}')

assert df[key_vars].dropna().shape[0] >= 30, 'Too few complete observations'
print('\nDescriptive statistics:')
df[key_vars].describe()

In [ ]:
# --- Write output ---
df.to_parquet(str(DATASET_PATH), index=False)
print(f'✓ Written: {DATASET_PATH}')
print(f'  Shape: {df.shape}')
print(f'  Columns: {list(df.columns)}')